# Path-cleaned version

All hard-coded working paths have been replaced with `/cityu/htcc/zliu-39/...`. Notebook outputs were cleared to avoid retaining previous absolute paths.


In [ ]:
# ============================================================
# Figure 1 | Step 0. 环境准备
# ============================================================

import os
import gzip
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.io
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

sc.settings.verbosity = 3
sc.set_figure_params(dpi=120, dpi_save=300, figsize=(5, 5))

# ------------------------------------------------------------
# 路径
# ------------------------------------------------------------
PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R"
DATA_DIR = f"{PROJECT_DIR}/R_loop_HCC/00_datacollection"

GSE151530_DIR = f"{DATA_DIR}/01_GSE151530"
GSE125449_DIR = f"{DATA_DIR}/03_GSE125449"

SET1_DIR = f"{GSE125449_DIR}/Set1"
SET2_DIR = f"{GSE125449_DIR}/Set2"

OUTDIR = f"{PROJECT_DIR}/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

print(OUTDIR)

In [ ]:
# ============================================================
# Figure 1 | Step 0. 环境准备
# ============================================================

import os
import gzip
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.io
import scipy.sparse as sp

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

sc.settings.verbosity = 3
sc.set_figure_params(dpi=120, dpi_save=300, figsize=(5, 5))

# ------------------------------------------------------------
# 路径
# ------------------------------------------------------------
PROJECT_DIR = "/cityu/htcc/zliu-39/Test_R"
DATA_DIR = f"{PROJECT_DIR}/R_loop_HCC/00_datacollection"

GSE151530_DIR = f"{DATA_DIR}/01_GSE151530"
GSE125449_DIR = f"{DATA_DIR}/03_GSE125449"

SET1_DIR = f"{GSE125449_DIR}/Set1"
SET2_DIR = f"{GSE125449_DIR}/Set2"

OUTDIR = f"{PROJECT_DIR}/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

print(OUTDIR)

In [ ]:
# ============================================================
# Figure 1 | Step 1. 通用读取函数
# ============================================================

def read_gz_table(path, sep="\t", header="infer"):
    return pd.read_csv(path, sep=sep, compression="gzip", header=header)


def read_10x_like_mtx(mtx_path, genes_path, barcodes_path=None, gene_col=1):
    """
    读取 mtx + genes/features + barcodes
    gene_col:
        0 = 第一列
        1 = 第二列，通常是 gene symbol
    """
    X = scipy.io.mmread(mtx_path).tocsr().T
    
    genes = pd.read_csv(
        genes_path,
        sep="\t",
        compression="gzip",
        header=None
    )
    
    gene_names = genes.iloc[:, gene_col].astype(str).values
    
    if barcodes_path is not None and os.path.exists(barcodes_path):
        barcodes = pd.read_csv(
            barcodes_path,
            sep="\t",
            compression="gzip",
            header=None
        ).iloc[:, 0].astype(str).values
    else:
        barcodes = [f"Cell_{i}" for i in range(X.shape[0])]
    
    adata = ad.AnnData(X=X)
    adata.var_names = gene_names
    adata.obs_names = barcodes
    
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    
    return adata


def qc_filter(adata, min_genes=500, min_counts=700, max_mt=20):
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=["mt"],
        percent_top=None,
        log1p=False,
        inplace=True
    )
    
    adata = adata[
        (adata.obs["n_genes_by_counts"] > min_genes) &
        (adata.obs["total_counts"] > min_counts) &
        (adata.obs["pct_counts_mt"] < max_mt)
    ].copy()
    
    return adata


def basic_preprocess(adata, n_top_genes=3000, n_pcs=30):
    adata = adata.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=n_top_genes,
        flavor="seurat"
    )
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=n_pcs)
    return adata

In [ ]:
# ============================================================
# Step 2 修正版：读取 GSE151530，并正确加入 S_ID / Sample / Type_raw
# ============================================================

import os
import pandas as pd
import scanpy as sc
import numpy as np

GSE151530_DIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/00_datacollection/01_GSE151530"

# ------------------------------------------------------------
# 1. 读取矩阵
# ------------------------------------------------------------

adata_151530 = read_10x_like_mtx(
    mtx_path=f"{GSE151530_DIR}/GSE151530_matrix.mtx.gz",
    genes_path=f"{GSE151530_DIR}/GSE151530_genes.tsv.gz",
    barcodes_path=None,
    gene_col=1
)

# ------------------------------------------------------------
# 2. 读取 metadata
# ------------------------------------------------------------

info_151530 = pd.read_csv(
    f"{GSE151530_DIR}/GSE151530_Info.txt.gz",
    sep="\t",
    compression="gzip"
)

print("GSE151530 info shape:", info_151530.shape)
print("GSE151530 info columns:")
print(info_151530.columns.tolist())
print(info_151530.head())

# ------------------------------------------------------------
# 3. 第三列是 cell barcode
# ------------------------------------------------------------

barcode_col = info_151530.columns[2]
print("Using barcode column:", barcode_col)

info_151530[barcode_col] = info_151530[barcode_col].astype(str)

if adata_151530.n_obs == info_151530.shape[0]:
    adata_151530.obs_names = info_151530[barcode_col].values
    adata_151530.obs_names_make_unique()
else:
    raise ValueError(
        f"细胞数不一致: matrix cells = {adata_151530.n_obs}, metadata rows = {info_151530.shape[0]}"
    )

# ------------------------------------------------------------
# 4. 挂载 metadata
# ------------------------------------------------------------

info_151530.index = info_151530[barcode_col].astype(str)

common_cells = adata_151530.obs_names.intersection(info_151530.index)

print("Matched cells:", len(common_cells))

adata_151530 = adata_151530[common_cells].copy()
adata_151530.obs = adata_151530.obs.join(info_151530.loc[common_cells])

# ------------------------------------------------------------
# 5. 标准化关键列
# ------------------------------------------------------------

adata_151530.obs["dataset"] = "GSE151530"
adata_151530.obs["batch"] = "GSE151530"

print("After metadata joining columns:")
print(adata_151530.obs.columns.tolist())

if "S_ID" not in adata_151530.obs.columns:
    raise ValueError("GSE151530 metadata 中没有 S_ID 列，请检查 info 文件")

adata_151530.obs["sample_id"] = adata_151530.obs["S_ID"].astype(str)

# 保存原始注释，不直接作为最终 Type
if "Type" in adata_151530.obs.columns:
    adata_151530.obs["Type_raw"] = adata_151530.obs["Type"].astype(str)
else:
    adata_151530.obs["Type_raw"] = "Unknown"

# 先清空最终 Type，后面用 marker 重新注释
adata_151530.obs["Type"] = "unclassified"

print("S_ID counts:")
print(adata_151530.obs["S_ID"].value_counts().head())

print("Original GSE151530 Type_raw counts:")
print(adata_151530.obs["Type_raw"].value_counts())

# ------------------------------------------------------------
# 6. QC
# ------------------------------------------------------------

adata_151530 = qc_filter(
    adata_151530,
    min_genes=500,
    min_counts=700,
    max_mt=20
)

print("GSE151530 after QC:")
print(adata_151530)

print("GSE151530 sample number:")
print(adata_151530.obs["sample_id"].nunique())

print("GSE151530 sample_id counts:")
print(adata_151530.obs["sample_id"].value_counts().head(20))

print("GSE151530 Type_raw after QC:")
print(adata_151530.obs["Type_raw"].value_counts())

In [ ]:
def read_GSE125449_set(set_dir, set_name):
    print(f"\n========== Reading {set_name} ==========")

    adata = read_10x_like_mtx(
        mtx_path=f"{set_dir}/GSE125449_{set_name}_matrix.mtx.gz",
        genes_path=f"{set_dir}/GSE125449_{set_name}_genes.tsv.gz",
        barcodes_path=f"{set_dir}/GSE125449_{set_name}_barcodes.tsv.gz",
        gene_col=1
    )

    samples = pd.read_csv(
        f"{set_dir}/GSE125449_{set_name}_samples.txt.gz",
        sep="\t",
        compression="gzip"
    )

    print("adata shape:", adata.shape)
    print("samples shape:", samples.shape)
    print("samples columns:", samples.columns.tolist())

    # 自动识别 barcode 列
    if "Cell.Barcode" in samples.columns:
        barcode_col = "Cell.Barcode"
    elif "Cell Barcode" in samples.columns:
        barcode_col = "Cell Barcode"
    elif "Cell_Barcode" in samples.columns:
        barcode_col = "Cell_Barcode"
    else:
        raise ValueError(f"找不到 barcode 列，目前列名是: {samples.columns.tolist()}")

    print("using barcode column:", barcode_col)
    print("adata barcode example:", adata.obs_names[:5].tolist())
    print("sample barcode example:", samples[barcode_col].head().tolist())

    # 统一 barcode 格式
    adata.obs_names = adata.obs_names.astype(str)
    samples[barcode_col] = samples[barcode_col].astype(str)

    adata.obs["barcode_raw"] = adata.obs_names
    adata.obs["barcode_clean"] = (
        adata.obs["barcode_raw"]
        .str.replace(r"-1$", "", regex=True)
    )

    samples["barcode_raw"] = samples[barcode_col]
    samples["barcode_clean"] = (
        samples["barcode_raw"]
        .str.replace(r"-1$", "", regex=True)
    )

    common = np.intersect1d(
        adata.obs["barcode_clean"].values,
        samples["barcode_clean"].values
    )

    print("common barcode number:", len(common))

    if len(common) == 0:
        raise ValueError(
            f"{set_name}: barcode 交集为 0，请检查 barcodes 和 samples 是否对应。"
        )

    samples_meta = samples.set_index("barcode_clean")

    adata.obs = adata.obs.join(
        samples_meta,
        on="barcode_clean",
        rsuffix="_sample"
    )

    adata = adata[adata.obs["Sample"].notna()].copy()

    adata.obs["dataset"] = "GSE125449"
    adata.obs["batch"] = set_name

    print("after metadata matching:", adata.shape)
    print(adata.obs[["Sample", "Type", "dataset", "batch"]].head())
    print(adata.obs["Sample"].value_counts().head())

    return adata

In [ ]:
adata_set1 = read_GSE125449_set(SET1_DIR, "Set1")
adata_set2 = read_GSE125449_set(SET2_DIR, "Set2")

adata_set1 = qc_filter(adata_set1, min_genes=500, min_counts=700, max_mt=20)
adata_set2 = qc_filter(adata_set2, min_genes=500, min_counts=700, max_mt=20)

print("Set1 after QC:", adata_set1)
print("Set2 after QC:", adata_set2)

In [ ]:
# ============================================================
# Figure 1 | Step 4.
# GSE125449 内部整合（Set1 + Set2）
# Stable Harmony integration workflow
# PDF-saving professional version
# ============================================================

import os
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import harmonypy as hm
import matplotlib.pyplot as plt

# ============================================================
# 0. Output directory
# ============================================================

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

os.makedirs(OUTDIR, exist_ok=True)

print("Output directory:")
print(OUTDIR)

# ============================================================
# 1. concat Set1 + Set2
# ============================================================

adata_125449 = ad.concat(
    [adata_set1, adata_set2],
    label="batch_source",
    keys=["Set1", "Set2"],
    join="outer",
    merge="same",
    index_unique="-"
)

adata_125449.var_names_make_unique()

print("\n========== Raw concatenated object ==========")
print(adata_125449)

# ============================================================
# 2. 保存 raw counts
# ============================================================

adata_125449.layers["counts"] = adata_125449.X.copy()

# ============================================================
# 3. Normalize
# ============================================================

print("\n========== Normalize ==========")

sc.pp.normalize_total(
    adata_125449,
    target_sum=1e4
)

sc.pp.log1p(adata_125449)

print("Normalization finished.")

# ============================================================
# 4. Highly Variable Genes
# ============================================================

print("\n========== Highly Variable Genes ==========")

sc.pp.highly_variable_genes(
    adata_125449,
    n_top_genes=3000,
    batch_key="batch",
    flavor="seurat_v3"
)

print(
    "HVG number:",
    adata_125449.var["highly_variable"].sum()
)

# 保存 HVG 信息
hvg_df = adata_125449.var.copy()

hvg_df.to_csv(
    f"{OUTDIR}/GSE125449_HVG_information.csv"
)

# 只保留 HVG
adata_125449 = adata_125449[
    :,
    adata_125449.var["highly_variable"]
].copy()

print("\n========== After HVG filtering ==========")
print(adata_125449)

# ============================================================
# 5. Scaling
# ============================================================

print("\n========== Scaling ==========")

sc.pp.scale(
    adata_125449,
    max_value=10
)

print("Scaling finished.")

# ============================================================
# 6. PCA
# ============================================================

print("\n========== PCA ==========")

sc.tl.pca(
    adata_125449,
    n_comps=50,
    svd_solver="arpack"
)

print("PCA embedding shape:")
print(adata_125449.obsm["X_pca"].shape)

# ============================================================
# PCA variance ratio plot
# ============================================================

fig, ax = plt.subplots(figsize=(5, 4))

sc.pl.pca_variance_ratio(
    adata_125449,
    n_pcs=50,
    log=True,
    show=False
)

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/GSE125449_PCA_variance_ratio.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/GSE125449_PCA_variance_ratio.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 7. Harmony integration
# ============================================================

print("\n========== Harmony integration ==========")

# PCA embedding
pca_embedding = adata_125449.obsm["X_pca"]

# metadata
meta_data = adata_125449.obs.copy()

# Run Harmony
ho = hm.run_harmony(
    pca_embedding,
    meta_data,
    vars_use=["batch"]
)

print("Original Harmony output shape:")
print(ho.Z_corr.shape)

# 自动判断是否需要 transpose
if ho.Z_corr.shape[0] == adata_125449.n_obs:
    harmony_embedding = ho.Z_corr
else:
    harmony_embedding = ho.Z_corr.T

print("Final Harmony embedding shape:")
print(harmony_embedding.shape)

# 保存 corrected PCs
adata_125449.obsm["X_pca_harmony"] = harmony_embedding

print("\nAvailable embeddings:")
print(adata_125449.obsm.keys())

# ============================================================
# 8. Neighbors
# ============================================================

print("\n========== Neighbors ==========")

sc.pp.neighbors(
    adata_125449,
    use_rep="X_pca_harmony",
    n_neighbors=25,
    n_pcs=30
)

print("Neighbors finished.")

# ============================================================
# 9. UMAP
# ============================================================

print("\n========== UMAP ==========")

sc.tl.umap(
    adata_125449,
    min_dist=0.35,
    spread=1.0,
    random_state=0
)

print("UMAP finished.")

# ============================================================
# 10. Leiden clustering
# ============================================================

print("\n========== Leiden clustering ==========")

sc.tl.leiden(
    adata_125449,
    resolution=0.5,
    key_added="leiden"
)

print("Leiden finished.")

# ============================================================
# 11. Save integrated object
# ============================================================

save_path = f"{OUTDIR}/adata_125449_integrated.h5ad"

adata_125449.write_h5ad(save_path)

print("\n========== Final object ==========")
print(adata_125449)

print("\nSaved to:")
print(save_path)

# ============================================================
# 12. Batch mixing QC UMAP
# ============================================================

print("\n========== Plot batch mixing ==========")

fig = sc.pl.umap(
    adata_125449,
    color="batch",
    frameon=False,
    size=10,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_batch_harmony.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_batch_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 13. Cell type QC UMAP
# ============================================================

print("\n========== Plot cell types ==========")

fig = sc.pl.umap(
    adata_125449,
    color="Type",
    frameon=False,
    legend_loc="on data",
    size=10,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_celltype_harmony.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_celltype_harmony.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 14. Leiden cluster UMAP
# ============================================================

print("\n========== Plot Leiden clusters ==========")

fig = sc.pl.umap(
    adata_125449,
    color="leiden",
    frameon=False,
    legend_loc="on data",
    size=10,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_leiden.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/Figure1_GSE125449_leiden.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 15. Save metadata
# ============================================================

adata_125449.obs.to_csv(
    f"{OUTDIR}/GSE125449_integrated_metadata.csv"
)

print("\n========== ALL FINISHED ==========")

In [ ]:
import gc
import os
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import harmonypy as hm

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# Step 5A. 重新构建 GSE125449 raw object
# 注意：不要用已经只剩 3000 HVG 的 adata_125449
# ============================================================

adata_125449_raw = ad.concat(
    [adata_set1, adata_set2],
    label="set",
    keys=["Set1", "Set2"],
    join="inner",
    merge="same",
    index_unique="-"
)

adata_125449_raw.obs["dataset"] = "GSE125449"
adata_125449_raw.obs["batch"] = adata_125449_raw.obs["set"].astype(str)
adata_125449_raw.obs["sample_id"] = adata_125449_raw.obs["Sample"].astype(str)

# 保留 GSE125449 原始注释
adata_125449_raw.obs["Type"] = adata_125449_raw.obs["Type"].astype(str)

# ============================================================
# Step 5B. 检查 GSE125449 metadata
# ============================================================

print("GSE125449 raw object:")
print(adata_125449_raw)

print("\nGSE125449 cell number:")
print(adata_125449_raw.n_obs)

print("\nGSE125449 sample number:")
print(adata_125449_raw.obs["sample_id"].nunique())

print("\nGSE125449 Type counts:")
print(adata_125449_raw.obs["Type"].value_counts())

print("\nGSE125449 sample examples:")
print(adata_125449_raw.obs["sample_id"].value_counts().head(20))

# ============================================================
# Step 5C. 确认 GSE151530 metadata 仍然存在
# 这里只检查，不合并
# ============================================================

adata_151530.obs["dataset"] = "GSE151530"
adata_151530.obs["batch"] = "GSE151530"
adata_151530.obs["sample_id"] = adata_151530.obs["S_ID"].astype(str)

if "Type_raw" not in adata_151530.obs.columns:
    if "Type" in adata_151530.obs.columns:
        adata_151530.obs["Type_raw"] = adata_151530.obs["Type"].astype(str)
    else:
        adata_151530.obs["Type_raw"] = "Unknown"

# 注意：这里先不要把 Type 当成最终注释
# 后面需要根据 marker / clustering 重新生成 Type

print("\nGSE151530 object before annotation:")
print(adata_151530)

print("\nGSE151530 sample number:")
print(adata_151530.obs["sample_id"].nunique())

print("\nGSE151530 Type_raw counts:")
print(adata_151530.obs["Type_raw"].value_counts())

print("\nGSE151530 current Type counts:")
if "Type" in adata_151530.obs.columns:
    print(adata_151530.obs["Type"].value_counts())
else:
    print("No Type column yet")

# ============================================================
# Step 5D. 保存检查点
# ============================================================

adata_125449_raw.write_h5ad(
    f"{OUTDIR}/Step5A_adata_125449_raw_for_reference.h5ad"
)

adata_151530.write_h5ad(
    f"{OUTDIR}/Step5B_adata_151530_before_annotation.h5ad"
)

gc.collect()

print("\n========== Step 5 finished ==========")
print("下一步：先对 GSE151530 单独做 normalize / PCA / UMAP / Leiden / marker，然后再按 GSE125449 的 Type 标准注释。")

In [ ]:
# ============================================================
# Step 6. GSE151530 单独预处理 + 聚类
# 目的：先给 GSE151530 做亚群注释，不急着合并 adata_all
# ============================================================

import gc
import os
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

# 如果你刚刚保存过这个文件，可以从检查点读取
# 否则如果内存中已经有 adata_151530，可以注释掉这一行
adata_151530 = sc.read_h5ad(f"{OUTDIR}/Step5B_adata_151530_before_annotation.h5ad")

print("Before normalize:")
print(adata_151530)

print("Sample number:")
print(adata_151530.obs["sample_id"].nunique())

print("Type_raw counts:")
print(adata_151530.obs["Type_raw"].value_counts())

# ------------------------------------------------------------
# 1. 保存 raw counts
# ------------------------------------------------------------

adata_151530.layers["counts"] = adata_151530.X.copy()

# ------------------------------------------------------------
# 2. Normalize + log1p
# ------------------------------------------------------------

sc.pp.normalize_total(
    adata_151530,
    target_sum=1e4
)

sc.pp.log1p(adata_151530)

# ------------------------------------------------------------
# 3. HVG
# ------------------------------------------------------------

sc.pp.highly_variable_genes(
    adata_151530,
    n_top_genes=3000,
    batch_key="sample_id",
    flavor="seurat_v3"
)

print("HVG number:", adata_151530.var["highly_variable"].sum())

adata_151530 = adata_151530[:, adata_151530.var["highly_variable"]].copy()

adata_151530.X = adata_151530.X.astype(np.float32)

print("After HVG:")
print(adata_151530)

# ------------------------------------------------------------
# 4. Scale + PCA
# ------------------------------------------------------------

sc.pp.scale(
    adata_151530,
    max_value=10
)

sc.tl.pca(
    adata_151530,
    n_comps=50,
    svd_solver="arpack"
)

# ------------------------------------------------------------
# 5. Neighbors + UMAP + Leiden
# ------------------------------------------------------------

sc.pp.neighbors(
    adata_151530,
    n_neighbors=25,
    n_pcs=30
)

sc.tl.umap(
    adata_151530,
    min_dist=0.35,
    spread=1.0,
    random_state=0
)

sc.tl.leiden(
    adata_151530,
    resolution=0.5,
    key_added="leiden"
)

print("Leiden counts:")
print(adata_151530.obs["leiden"].value_counts().sort_index())

# ------------------------------------------------------------
# 6. 画图检查
# ------------------------------------------------------------

fig = sc.pl.umap(
    adata_151530,
    color="sample_id",
    frameon=False,
    size=8,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_sample_id.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_sample_id.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

fig = sc.pl.umap(
    adata_151530,
    color="leiden",
    frameon=False,
    legend_loc="on data",
    size=8,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_leiden.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_leiden.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 如果 Type_raw 有信息，也画一下
fig = sc.pl.umap(
    adata_151530,
    color="Type_raw",
    frameon=False,
    size=8,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_Type_raw.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/GSE151530_UMAP_Type_raw.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 7. 保存
# ------------------------------------------------------------

adata_151530.write_h5ad(
    f"{OUTDIR}/Step6_GSE151530_clustered_before_annotation.h5ad"
)

gc.collect()

print("\n========== Step 6 finished ==========")
print("下一步：根据 marker genes 给 GSE151530 的 leiden cluster 注释成 B cell / CAF / HPC-like / Malignant cell / T cell / TAM / TEC / unclassified")

In [ ]:
# ============================================================
# Step 7. GSE151530 marker genes 检查
# 目的：根据 marker 给 GSE151530 的 leiden cluster 做细胞亚群注释
# ============================================================

import os
import gc
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

adata_151530 = sc.read_h5ad(
    f"{OUTDIR}/Step6_GSE151530_clustered_before_annotation.h5ad"
)

print("Loaded GSE151530 clustered object:")
print(adata_151530)

print("Leiden clusters:")
print(adata_151530.obs["leiden"].value_counts().sort_index())

# ------------------------------------------------------------
# 1. 定义参考 marker
# 参照 GSE125449 的大类注释体系
# ------------------------------------------------------------

marker_dict = {
    "T cell": [
        "CD3D", "CD3E", "CD2", "TRAC", "IL7R", "NKG7", "GNLY"
    ],
    "B cell": [
        "MS4A1", "CD79A", "CD79B", "MS4A1", "MZB1", "JCHAIN"
    ],
    "TAM": [
        "LYZ", "C1QA", "C1QB", "C1QC", "CD68", "CD163", "MARCO"
    ],
    "CAF": [
        "COL1A1", "COL1A2", "DCN", "LUM", "ACTA2", "TAGLN", "RGS5"
    ],
    "TEC": [
        "PECAM1", "VWF", "KDR", "ENG", "EMCN", "RAMP2"
    ],
    "HPC-like": [
        "KRT19", "EPCAM", "SOX9", "PROM1", "MUC1"
    ],
    "Malignant cell": [
        "ALB", "AFP", "APOA1", "APOA2", "TTR", "KRT8", "KRT18", "EPCAM"
    ]
}

# 只保留数据里存在的 marker
marker_dict_use = {}

for ct, genes in marker_dict.items():
    genes_use = [g for g in genes if g in adata_151530.var_names]
    if len(genes_use) > 0:
        marker_dict_use[ct] = genes_use

print("Available marker genes:")
for ct, genes in marker_dict_use.items():
    print(ct, genes)

# ------------------------------------------------------------
# 2. Dotplot：看每个 leiden cluster 的 marker 表达
# ------------------------------------------------------------

fig = sc.pl.dotplot(
    adata_151530,
    marker_dict_use,
    groupby="leiden",
    standard_scale="var",
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/GSE151530_marker_dotplot_by_leiden.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/GSE151530_marker_dotplot_by_leiden.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. rank_genes_groups：计算每个 cluster 的 marker
# ------------------------------------------------------------

sc.tl.rank_genes_groups(
    adata_151530,
    groupby="leiden",
    method="wilcoxon",
    pts=True,
    key_added="rank_genes_leiden"
)

# 保存 top markers
marker_tables = []

for cluster in adata_151530.obs["leiden"].cat.categories:
    df = sc.get.rank_genes_groups_df(
        adata_151530,
        group=cluster,
        key="rank_genes_leiden"
    )
    df["cluster"] = cluster
    marker_tables.append(df)

markers_df = pd.concat(marker_tables, axis=0)

markers_df.to_csv(
    f"{OUTDIR}/GSE151530_leiden_rank_genes.csv",
    index=False
)

# 输出每个 cluster top 20 marker
for cluster in adata_151530.obs["leiden"].cat.categories:
    print("\n========== Cluster", cluster, "top markers ==========")
    tmp = markers_df[markers_df["cluster"] == cluster].head(20)
    print(tmp[["names", "scores", "logfoldchanges", "pvals_adj"]])

# ------------------------------------------------------------
# 4. UMAP 上看关键 marker
# ------------------------------------------------------------

marker_to_plot = [
    "CD3D", "CD3E", "MS4A1", "CD79A",
    "LYZ", "C1QA", "COL1A1", "DCN",
    "PECAM1", "VWF", "KRT19", "EPCAM",
    "ALB", "AFP", "KRT8", "KRT18"
]

marker_to_plot = [g for g in marker_to_plot if g in adata_151530.var_names]

if len(marker_to_plot) > 0:
    fig = sc.pl.umap(
        adata_151530,
        color=marker_to_plot,
        frameon=False,
        size=8,
        ncols=4,
        return_fig=True
    )

    fig.savefig(
        f"{OUTDIR}/GSE151530_marker_feature_umap.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        f"{OUTDIR}/GSE151530_marker_feature_umap.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# ------------------------------------------------------------
# 5. 保存
# ------------------------------------------------------------

adata_151530.write_h5ad(
    f"{OUTDIR}/Step7_GSE151530_marker_checked.h5ad"
)

gc.collect()

print("\n========== Step 7 finished ==========")
print("请查看 GSE151530_marker_dotplot_by_leiden.png 和 GSE151530_leiden_rank_genes.csv，然后根据每个 cluster 的 marker 手动指定 Type。")

In [ ]:
# ============================================================
# Step 8. 给 GSE151530 leiden cluster 做最终注释
# ============================================================

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

adata_151530 = sc.read_h5ad(
    f"{OUTDIR}/Step7_GSE151530_marker_checked.h5ad"
)

# ------------------------------------------------------------
# cluster -> cell type
# ------------------------------------------------------------

cluster_to_type = {
    "0": "T cell",
    "1": "T cell",
    "2": "TAM",
    "3": "Malignant cell",
    "4": "T cell",
    "5": "TEC",
    "6": "Malignant cell",
    "7": "T cell",
    "8": "Malignant cell",
    "9": "CAF",
    "10": "T cell",
    "11": "HPC-like",
    "12": "B cell",
    "13": "Malignant cell",
    "14": "Malignant cell",
    "15": "HPC-like",
    "16": "T cell",
    "17": "T cell",
    "18": "T cell",
    "19": "TAM",
    "20": "T cell",
    "21": "B cell",
    "22": "Malignant cell",
    "23": "HPC-like",
    "24": "HPC-like",
    "25": "HPC-like",
    "26": "B cell"
}

# ------------------------------------------------------------
# 添加最终 Type
# ------------------------------------------------------------

adata_151530.obs["Type"] = (
    adata_151530.obs["leiden"]
    .astype(str)
    .map(cluster_to_type)
)

adata_151530.obs["Type"] = (
    adata_151530.obs["Type"]
    .fillna("unclassified")
)

print("Final Type counts:")
print(adata_151530.obs["Type"].value_counts())

# ------------------------------------------------------------
# 画最终注释 UMAP
# ------------------------------------------------------------

fig = sc.pl.umap(
    adata_151530,
    color="Type",
    frameon=False,
    legend_loc="on data",
    size=8,
    return_fig=True
)

fig.savefig(
    f"{OUTDIR}/GSE151530_final_Type_annotation.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{OUTDIR}/GSE151530_final_Type_annotation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 保存最终注释版本
# ------------------------------------------------------------

adata_151530.write_h5ad(
    f"{OUTDIR}/Step8_GSE151530_final_annotated.h5ad"
)

print("\n========== GSE151530 annotation finished ==========")

In [ ]:
# ============================================================
# Step 9 low-memory final fixed version
# 重新合并 GSE151530_final + GSE125449_raw
# Normalize / PCA / Harmony
# 完全跳过 HVG
# ============================================================

import gc
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import harmonypy as hm
import scipy.sparse as sp

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# 1. 读取 GSE151530 final
# ------------------------------------------------------------

adata_151530_final = sc.read_h5ad(
    f"{OUTDIR}/Step8_GSE151530_final_annotated.h5ad"
)

adata_151530_final.obs["dataset"] = "GSE151530"
adata_151530_final.obs["batch"] = "GSE151530"
adata_151530_final.obs["sample_id"] = adata_151530_final.obs["S_ID"].astype(str)
adata_151530_final.obs["Type"] = adata_151530_final.obs["Type"].astype(str)

# ------------------------------------------------------------
# 2. 重新构建 GSE125449 raw
# ------------------------------------------------------------

adata_125449_raw = ad.concat(
    [adata_set1, adata_set2],
    label="set",
    keys=["Set1", "Set2"],
    join="inner",
    merge="same",
    index_unique="-"
)

adata_125449_raw.obs["dataset"] = "GSE125449"
adata_125449_raw.obs["batch"] = adata_125449_raw.obs["set"].astype(str)
adata_125449_raw.obs["sample_id"] = adata_125449_raw.obs["Sample"].astype(str)
adata_125449_raw.obs["Type"] = adata_125449_raw.obs["Type"].astype(str)

# ------------------------------------------------------------
# 3. 只保留必要 metadata
# ------------------------------------------------------------

keep_obs = ["dataset", "batch", "sample_id", "Type"]

adata_151530_final.obs = adata_151530_final.obs[keep_obs].copy()
adata_125449_raw.obs = adata_125449_raw.obs[keep_obs].copy()

# 确保 X 是稀疏矩阵 + float32
if not sp.issparse(adata_151530_final.X):
    adata_151530_final.X = sp.csr_matrix(adata_151530_final.X)

if not sp.issparse(adata_125449_raw.X):
    adata_125449_raw.X = sp.csr_matrix(adata_125449_raw.X)

adata_151530_final.X = adata_151530_final.X.astype(np.float32)
adata_125449_raw.X = adata_125449_raw.X.astype(np.float32)

# ------------------------------------------------------------
# 4. 合并两个数据集
# ------------------------------------------------------------

adata_all = ad.concat(
    [adata_151530_final, adata_125449_raw],
    label="dataset_merge",
    keys=["GSE151530", "GSE125449"],
    join="inner",
    merge="same",
    index_unique="-"
)

adata_all.var_names_make_unique()

adata_all.obs["sample_id"] = adata_all.obs["sample_id"].astype(str)
adata_all.obs["sample_id"] = adata_all.obs["sample_id"].replace(
    ["nan", "None", "", "Unknown"],
    np.nan
)

print("Merged object:")
print(adata_all)

print("\nDataset counts:")
print(adata_all.obs["dataset"].value_counts())

print("\nSample number by dataset:")
print(adata_all.obs.groupby("dataset", observed=True)["sample_id"].nunique())

print("\nType counts by dataset:")
print(pd.crosstab(adata_all.obs["dataset"], adata_all.obs["Type"]))

del adata_151530_final, adata_125449_raw
gc.collect()

# ------------------------------------------------------------
# 5. Normalize + log1p
# ------------------------------------------------------------

sc.pp.normalize_total(
    adata_all,
    target_sum=1e4
)

sc.pp.log1p(adata_all)

# ------------------------------------------------------------
# 6. 清理 NaN / Inf
# 注意：这里不做 HVG
# ------------------------------------------------------------

print("\nSkip HVG filtering completely.")
print("Gene number used for PCA:", adata_all.n_vars)

if sp.issparse(adata_all.X):
    adata_all.X.data = np.nan_to_num(
        adata_all.X.data,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )
else:
    adata_all.X = np.nan_to_num(
        adata_all.X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

adata_all.X = adata_all.X.astype(np.float32)

gc.collect()

print("After cleaning:")
print(adata_all)

# ------------------------------------------------------------
# 7. PCA
# 不运行 scale，避免 dense 爆内存
# ------------------------------------------------------------

sc.tl.pca(
    adata_all,
    n_comps=50,
    svd_solver="arpack"
)

print("\nPCA shape:")
print(adata_all.obsm["X_pca"].shape)

# ------------------------------------------------------------
# 8. Harmony integration
# ------------------------------------------------------------

ho = hm.run_harmony(
    data_mat=adata_all.obsm["X_pca"].astype(np.float32),
    meta_data=adata_all.obs,
    vars_use=["dataset"]
)

print("\nOriginal Harmony output shape:")
print(ho.Z_corr.shape)

if ho.Z_corr.shape[0] == adata_all.n_obs:
    harmony_embedding = ho.Z_corr
else:
    harmony_embedding = ho.Z_corr.T

if harmony_embedding.shape[0] != adata_all.n_obs:
    raise ValueError(
        f"Harmony 维度错误: {harmony_embedding.shape}, "
        f"应该是 ({adata_all.n_obs}, n_pcs)"
    )

adata_all.obsm["X_pca_harmony"] = harmony_embedding.astype(np.float32)

print("Harmony shape:")
print(adata_all.obsm["X_pca_harmony"].shape)

# ------------------------------------------------------------
# 9. 保存
# ------------------------------------------------------------

save_path = f"{OUTDIR}/Step9D_adata_all_final_annotation_Harmony_low_memory.h5ad"

adata_all.write_h5ad(save_path)

del ho, harmony_embedding
gc.collect()

print("\n========== Step 9 low-memory finished ==========")
print("Saved to:")
print(save_path)

In [ ]:
# ============================================================
# Figure 1 | Step 10. 统一细胞类型注释
# ============================================================

import scanpy as sc
import pandas as pd

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step9D_adata_all_final_annotation_Harmony_low_memory.h5ad"
)

print("Original Type counts:")
print(adata_all.obs["Type"].value_counts(dropna=False))

celltype_map = {
    "B cell": "B cell",
    "T cell": "T cell",
    "T/NK": "T cell",
    "TAM": "TAM",
    "Macrophage": "TAM",
    "CAF": "CAF",
    "TEC": "TEC",
    "Endothelial": "TEC",
    "HPC-like": "HPC-like",
    "HPC−like": "HPC-like",
    "Malignant cell": "Malignant cell",
    "Tumor": "Malignant cell",
    "unclassified": "unclassified",
    "Unknown": "unclassified"
}

adata_all.obs["celltype"] = (
    adata_all.obs["Type"]
    .astype(str)
    .map(celltype_map)
)

adata_all.obs["celltype"] = adata_all.obs["celltype"].fillna("unclassified")

adata_all.obs["celltype"] = adata_all.obs["celltype"].replace(
    ["nan", "None", "NA", ""],
    "unclassified"
)

print("\nFinal celltype counts:")
print(adata_all.obs["celltype"].value_counts())

print("\nCelltype counts by dataset:")
print(pd.crosstab(adata_all.obs["dataset"], adata_all.obs["celltype"]))

adata_all.write_h5ad(
    f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"
)

In [ ]:
# ============================================================
# Figure 1 | Step 10. 统一细胞类型注释
# ============================================================

import scanpy as sc
import pandas as pd

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step9D_adata_all_final_annotation_Harmony_low_memory.h5ad"
)

print("Original Type counts:")
print(adata_all.obs["Type"].value_counts(dropna=False))

celltype_map = {
    "B cell": "B cell",
    "T cell": "T cell",
    "T/NK": "T cell",
    "TAM": "TAM",
    "Macrophage": "TAM",
    "CAF": "CAF",
    "TEC": "TEC",
    "Endothelial": "TEC",
    "HPC-like": "HPC-like",
    "HPC−like": "HPC-like",
    "Malignant cell": "Malignant cell",
    "Tumor": "Malignant cell",
    "unclassified": "unclassified",
    "Unknown": "unclassified"
}

adata_all.obs["celltype"] = (
    adata_all.obs["Type"]
    .astype(str)
    .map(celltype_map)
)

adata_all.obs["celltype"] = adata_all.obs["celltype"].fillna("unclassified")

adata_all.obs["celltype"] = adata_all.obs["celltype"].replace(
    ["nan", "None", "NA", ""],
    "unclassified"
)

print("\nFinal celltype counts:")
print(adata_all.obs["celltype"].value_counts())

print("\nCelltype counts by dataset:")
print(pd.crosstab(adata_all.obs["dataset"], adata_all.obs["celltype"]))

adata_all.write_h5ad(
    f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"
)

In [ ]:
# ============================================================
# Step 10. Final unified celltype annotation
# ============================================================

import scanpy as sc
import pandas as pd

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

# ------------------------------------------------------------
# 1. 读取 Harmony 后对象
# ------------------------------------------------------------

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step9D_adata_all_final_annotation_Harmony_low_memory.h5ad"
)

print("Original Type counts:")
print(adata_all.obs["Type"].value_counts(dropna=False))

# ------------------------------------------------------------
# 2. 统一细胞类型命名
# ------------------------------------------------------------

celltype_map = {
    "B cell": "B cell",
    "T cell": "T cell",
    "T/NK": "T cell",
    "TAM": "TAM",
    "Macrophage": "TAM",
    "CAF": "CAF",
    "TEC": "TEC",
    "Endothelial": "TEC",
    "HPC-like": "HPC-like",
    "HPC−like": "HPC-like",
    "Tumor": "Malignant cell",
    "Malignant cell": "Malignant cell",
    "Unknown": "unclassified",
    "unclassified": "unclassified"
}

adata_all.obs["celltype"] = (
    adata_all.obs["Type"]
    .astype(str)
    .map(celltype_map)
)

adata_all.obs["celltype"] = adata_all.obs["celltype"].fillna(
    "unclassified"
)

adata_all.obs["celltype"] = adata_all.obs["celltype"].replace(
    ["nan", "None", "", "NA"],
    "unclassified"
)

# ------------------------------------------------------------
# 3. 检查结果
# ------------------------------------------------------------

print("\nFinal celltype counts:")
print(adata_all.obs["celltype"].value_counts())

print("\nCelltype counts by dataset:")
print(
    pd.crosstab(
        adata_all.obs["dataset"],
        adata_all.obs["celltype"]
    )
)

# ------------------------------------------------------------
# 4. 保存最终对象
# ------------------------------------------------------------

save_path = f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"

adata_all.write_h5ad(save_path)

print("\nSaved to:")
print(save_path)

In [ ]:
# ============================================================
# Figure 1 | Step 11. UMAP 加虚线圈函数
# 可选：只用于美化 UMAP，不影响数据分析
# ============================================================

import numpy as np
import pandas as pd
from matplotlib.patches import Ellipse

def add_dashed_ellipse_by_group(
    ax,
    adata,
    groupby,
    groups=None,
    basis="X_umap",
    scale=2.4,
    min_cells=50,
    lw=1.2
):
    emb = adata.obsm[basis]

    df = pd.DataFrame({
        "x": emb[:, 0],
        "y": emb[:, 1],
        "group": adata.obs[groupby].astype(str).values
    })

    if groups is None:
        groups = df["group"].unique()

    for g in groups:
        sub = df[df["group"] == g]

        if sub.shape[0] < min_cells:
            continue

        x = sub["x"].values
        y = sub["y"].values

        cx, cy = np.median(x), np.median(y)
        w = np.percentile(x, 95) - np.percentile(x, 5)
        h = np.percentile(y, 95) - np.percentile(y, 5)

        ell = Ellipse(
            xy=(cx, cy),
            width=w * scale,
            height=h * scale,
            angle=0,
            fill=False,
            linestyle="--",
            linewidth=lw,
            edgecolor="black",
            alpha=0.8
        )

        ax.add_patch(ell)
        ax.text(cx, cy, g, fontsize=8, ha="center", va="center")

    return ax

In [ ]:
print("adata_all GSE151530 sample_id:")

tmp = adata_all.obs.loc[
    adata_all.obs["dataset"] == "GSE151530",
    "sample_id"
]

print(tmp.value_counts().head())

print("\nUnique sample number:")
print(tmp.nunique())

In [ ]:
import scanpy as sc
import pandas as pd

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step9D_adata_all_final_annotation_Harmony_low_memory.h5ad"
)

print("dataset counts:")
print(adata_all.obs["dataset"].value_counts())

print("\nsample number by dataset:")
print(adata_all.obs.groupby("dataset", observed=True)["sample_id"].nunique())

print("\ncelltype / Type counts by dataset:")
print(pd.crosstab(adata_all.obs["dataset"], adata_all.obs["Type"]))

print("\nGSE151530 sample examples:")
print(
    adata_all.obs.loc[
        adata_all.obs["dataset"] == "GSE151530",
        "sample_id"
    ].value_counts().head(20)
)

In [ ]:
# ============================================================
# Figure 1A | Final version
# 样本来源 + malignant cell 数量 + cell proportion
# 使用 Step10_adata_all_final_celltype.h5ad
# ============================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# 1. 读取最终对象
# ------------------------------------------------------------

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"
)

adata_all.obs["dataset"] = adata_all.obs["dataset"].astype(str)
adata_all.obs["sample_id"] = adata_all.obs["sample_id"].astype(str)

adata_all.obs["sample_id"] = adata_all.obs["sample_id"].replace(
    ["nan", "None", "", "Unknown"],
    np.nan
)

for col in ["sample_id", "dataset", "celltype"]:
    if col not in adata_all.obs.columns:
        raise ValueError(f"adata_all.obs 中没有 {col} 列")

print("sample number by dataset:")
print(adata_all.obs.groupby("dataset", observed=True)["sample_id"].nunique())

print("\nCelltype counts by dataset:")
print(pd.crosstab(adata_all.obs["dataset"], adata_all.obs["celltype"]))

# ------------------------------------------------------------
# 2. 准备绘图数据
# ------------------------------------------------------------

obs_df = adata_all.obs[
    ["sample_id", "dataset", "celltype"]
].copy()

obs_df = obs_df.dropna(subset=["sample_id", "celltype"])

obs_df["sample_id"] = obs_df["sample_id"].astype(str)
obs_df["dataset"] = obs_df["dataset"].astype(str)
obs_df["celltype"] = obs_df["celltype"].astype(str)

total_df = (
    obs_df
    .groupby(["sample_id", "dataset"], observed=True)
    .size()
    .reset_index(name="total")
)

prop_df = (
    obs_df
    .groupby(["sample_id", "dataset", "celltype"], observed=True)
    .size()
    .reset_index(name="n")
)

prop_df = prop_df.merge(
    total_df,
    on=["sample_id", "dataset"],
    how="left"
)

prop_df["prop"] = prop_df["n"] / prop_df["total"] * 100

# ------------------------------------------------------------
# 3. malignant cell 数量
# ------------------------------------------------------------

malignant_label = "Malignant cell"

mal_df = (
    prop_df[prop_df["celltype"] == malignant_label]
    [["sample_id", "dataset", "n"]]
    .rename(columns={"n": "malignant_n"})
)

sample_df = total_df.merge(
    mal_df,
    on=["sample_id", "dataset"],
    how="left"
)

sample_df["malignant_n"] = sample_df["malignant_n"].fillna(0)
sample_df["log2_malignant_n"] = np.log2(sample_df["malignant_n"] + 1)

# ------------------------------------------------------------
# 4. 样本排序
# ------------------------------------------------------------

dataset_order = ["GSE125449", "GSE151530"]

sample_df["dataset"] = pd.Categorical(
    sample_df["dataset"],
    categories=dataset_order,
    ordered=True
)

sample_df = sample_df.sort_values(
    ["dataset", "log2_malignant_n"],
    ascending=[True, False]
).reset_index(drop=True)

sample_order = sample_df["sample_id"].tolist()

print("\nFinal sample number:", len(sample_order))
print(sample_df["dataset"].value_counts())

# ------------------------------------------------------------
# 5. cell proportion matrix
# ------------------------------------------------------------

prop_pivot = prop_df.pivot_table(
    index="sample_id",
    columns="celltype",
    values="prop",
    fill_value=0,
    aggfunc="sum"
)

prop_pivot = prop_pivot.reindex(sample_order).fillna(0)

celltype_order = [
    "B cell",
    "CAF",
    "HPC-like",
    "Malignant cell",
    "T cell",
    "TAM",
    "TEC",
    "unclassified"
]

celltype_order = [x for x in celltype_order if x in prop_pivot.columns]
prop_pivot = prop_pivot[celltype_order]

# ------------------------------------------------------------
# 6. 颜色
# ------------------------------------------------------------

dataset_colors = {
    "GSE125449": "#D98C92",
    "GSE151530": "#A9C2DD"
}

celltype_colors = {
    "B cell": "#80B1D3",
    "CAF": "#FFFFB3",
    "HPC-like": "#FCCDE5",
    "Malignant cell": "#8DD3C7",
    "T cell": "#B3DE69",
    "TAM": "#7BA6C2",
    "TEC": "#FDB462",
    "unclassified": "#A6D854"
}

# ------------------------------------------------------------
# 7. 绘图
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3, 1,
    figsize=(max(14, len(sample_order) * 0.23), 7.5),
    gridspec_kw={"height_ratios": [0.35, 1.1, 3.2]},
    sharex=True
)

x = np.arange(len(sample_order))

dataset_series = (
    sample_df
    .set_index("sample_id")
    .reindex(sample_order)["dataset"]
    .astype(str)
)

bar_colors = dataset_series.map(dataset_colors).fillna("#CCCCCC")

axes[0].bar(
    x,
    np.ones(len(sample_order)),
    color=bar_colors,
    width=0.9
)

axes[0].set_ylabel("Data\nSource", rotation=0, ha="right", va="center")
axes[0].set_yticks([])
axes[0].spines[:].set_visible(False)

log_vals = (
    sample_df
    .set_index("sample_id")
    .reindex(sample_order)["log2_malignant_n"]
    .values
)

axes[1].plot(
    x,
    log_vals,
    color="black",
    marker="o",
    markersize=3,
    linewidth=1.0
)

axes[1].set_ylabel("log2(Malignant cell no.)")
axes[1].spines[["top", "right"]].set_visible(False)

bottom = np.zeros(len(sample_order))

for ct in prop_pivot.columns:
    vals = prop_pivot[ct].values
    axes[2].bar(
        x,
        vals,
        bottom=bottom,
        label=ct,
        color=celltype_colors.get(ct, "#CCCCCC"),
        width=0.9
    )
    bottom += vals

axes[2].set_ylabel("Cell proportion (%)")
axes[2].set_ylim(0, 100)
axes[2].set_xticks(x)
axes[2].set_xticklabels(sample_order, rotation=90, fontsize=7)
axes[2].spines[["top", "right"]].set_visible(False)

# ------------------------------------------------------------
# 8. legend
# ------------------------------------------------------------

dataset_handles = [
    plt.Line2D(
        [0], [0],
        marker="s",
        color="w",
        label=k,
        markerfacecolor=v,
        markersize=10
    )
    for k, v in dataset_colors.items()
]

celltype_handles = [
    plt.Line2D(
        [0], [0],
        marker="s",
        color="w",
        label=k,
        markerfacecolor=celltype_colors.get(k, "#CCCCCC"),
        markersize=10
    )
    for k in prop_pivot.columns
]

axes[2].legend(
    handles=dataset_handles + celltype_handles,
    bbox_to_anchor=(1.02, 1.05),
    loc="upper left",
    frameon=False,
    fontsize=9
)

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1A_sample_cell_proportion_final.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1A_sample_cell_proportion_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# Figure 1B. UMAP by dataset
# ============================================================

import scanpy as sc
import matplotlib.pyplot as plt

OUTDIR = "/cityu/htcc/zliu-39/Test_R/R_loop_HCC/01_Figure1_python"

adata_all = sc.read_h5ad(
    f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"
)

# ============================================================
# Run neighbors + UMAP if not exist
# ============================================================

if "X_umap" not in adata_all.obsm.keys():

    print("Running neighbors + UMAP...")

    sc.pp.neighbors(
        adata_all,
        use_rep="X_pca_harmony",
        n_neighbors=25,
        n_pcs=30
    )

    sc.tl.umap(
        adata_all,
        min_dist=0.35,
        spread=1.0,
        random_state=0
    )

    adata_all.write_h5ad(
        f"{OUTDIR}/Step10_adata_all_final_celltype.h5ad"
    )

# ============================================================
# 1. UMAP WITH LEGEND
# ============================================================

fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    adata_all,
    color="dataset",
    ax=ax,
    show=False,
    size=6,
    frameon=False,
    legend_loc="right margin"   # 保留legend
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1B_UMAP_dataset_withLegend.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1B_UMAP_dataset_withLegend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ============================================================
# 2. UMAP WITHOUT LEGEND
# ============================================================

fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    adata_all,
    color="dataset",
    ax=ax,
    show=False,
    size=6,
    frameon=False,
    legend_loc=None   # 去除legend
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1B_UMAP_dataset_noLegend.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1B_UMAP_dataset_noLegend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("UMAP figures saved successfully!")

In [ ]:
# ============================================================
# Figure 1C. UMAP by major cell type
# ============================================================

# ============================================================
# 1. UMAP WITH LEGEND
# ============================================================

fig, ax = plt.subplots(figsize=(5.5, 5.5))

sc.pl.umap(
    adata_all,
    color="celltype",
    ax=ax,
    show=False,
    size=5,
    frameon=False,
    legend_loc="right margin"
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1C_UMAP_celltype_withLegend.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1C_UMAP_celltype_withLegend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 2. UMAP WITHOUT LEGEND
# ============================================================

fig, ax = plt.subplots(figsize=(5.5, 5.5))

sc.pl.umap(
    adata_all,
    color="celltype",
    ax=ax,
    show=False,
    size=5,
    frameon=False,
    legend_loc=None
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1C_UMAP_celltype_noLegend.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1C_UMAP_celltype_noLegend.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure 1C UMAP figures saved successfully!")

In [ ]:
# ============================================================
# Figure 1D. CNV score UMAP
# Using epithelial/malignant markers to estimate CNV-like score
# ============================================================

cnv_like_genes = [
    "EPCAM", "KRT8", "KRT18", "KRT19", "ALB", "AFP",
    "GPC3", "MKI67", "TOP2A", "UBE2C"
]

cnv_like_genes = [g for g in cnv_like_genes if g in adata_all.var_names]

print("Genes used for CNV_score:", cnv_like_genes)

sc.tl.score_genes(
    adata_all,
    gene_list=cnv_like_genes,
    score_name="CNV_score"
)

fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    adata_all,
    color="CNV_score",
    cmap="Purples",
    ax=ax,
    show=False,
    size=5,
    frameon=False,
    colorbar_loc="right"
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()

plt.savefig(
    f"{OUTDIR}/Figure1D_UMAP_CNV_score.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUTDIR}/Figure1D_UMAP_CNV_score.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Figure 1D CNV score UMAP saved successfully!")

In [ ]:
# ============================================================
# Figure 1E. UMAP malignant-centered annotation
# 可以只显示 GSE125449/GSE151530 HCC malignant 相关细胞
# ============================================================

fig, ax = plt.subplots(figsize=(5.5, 5.5))

sc.pl.umap(
    adata_all,
    color="celltype",
    groups=[
        "B cell", "HPC-like", "Malignant cell",
        "T cell", "TAM", "unclassified"
    ],
    ax=ax,
    show=False,
    size=5,
    frameon=False,
    legend_loc="right margin"
)

add_dashed_ellipse_by_group(
    ax=ax,
    adata=adata_all,
    groupby="celltype",
    groups=[
        "B cell", "HPC-like", "Malignant cell",
        "T cell", "TAM", "unclassified"
    ],
    scale=1.4,
    min_cells=80
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig(f"{OUTDIR}/Figure1E_UMAP_selected_celltype.pdf")
plt.savefig(f"{OUTDIR}/Figure1E_UMAP_selected_celltype.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Figure 1F. T cell subset annotation: CD4 T / CD8 T / Treg
# ============================================================

t_cells = adata_all[adata_all.obs["celltype"].isin(["T cell"])].copy()

# T cell 子群打分
gene_sets_t = {
    "CD4_T_score": ["CD4", "IL7R", "LTB", "CCR7"],
    "CD8_T_score": ["CD8A", "CD8B", "NKG7", "CST7", "GZMB"],
    "Treg_score": ["FOXP3", "IL2RA", "CTLA4", "TIGIT"]
}

for score_name, genes in gene_sets_t.items():
    genes = [g for g in genes if g in t_cells.var_names]
    if len(genes) > 0:
        sc.tl.score_genes(t_cells, genes, score_name=score_name)

score_cols = [c for c in gene_sets_t.keys() if c in t_cells.obs.columns]
t_cells.obs["T_subtype"] = t_cells.obs[score_cols].idxmax(axis=1)
t_cells.obs["T_subtype"] = t_cells.obs["T_subtype"].replace({
    "CD4_T_score": "CD4 T",
    "CD8_T_score": "CD8 T",
    "Treg_score": "Treg"
})

sc.pp.neighbors(t_cells, use_rep="X_pca_harmony" if "X_pca_harmony" in t_cells.obsm else "X_pca")
sc.tl.umap(t_cells)

fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    t_cells,
    color="T_subtype",
    ax=ax,
    show=False,
    size=6,
    frameon=False,
    legend_loc="right margin"
)

add_dashed_ellipse_by_group(
    ax=ax,
    adata=t_cells,
    groupby="T_subtype",
    groups=["CD4 T", "CD8 T", "Treg"],
    scale=1.4,
    min_cells=30
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig(f"{OUTDIR}/Figure1F_Tcell_subtype_UMAP.pdf")
plt.savefig(f"{OUTDIR}/Figure1F_Tcell_subtype_UMAP.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Figure 1G. TAM subset annotation: M1 / M2
# ============================================================

tam = adata_all[adata_all.obs["celltype"].isin(["TAM"])].copy()

gene_sets_mac = {
    "M1_score": ["IL1B", "TNF", "CXCL9", "CXCL10", "FCGR3A"],
    "M2_score": ["CD163", "MRC1", "MSR1", "APOE", "C1QC"]
}

for score_name, genes in gene_sets_mac.items():
    genes = [g for g in genes if g in tam.var_names]
    if len(genes) > 0:
        sc.tl.score_genes(tam, genes, score_name=score_name)

tam.obs["Macrophage_subtype"] = np.where(
    tam.obs["M1_score"] >= tam.obs["M2_score"],
    "M1",
    "M2"
)

sc.pp.neighbors(tam, use_rep="X_pca_harmony" if "X_pca_harmony" in tam.obsm else "X_pca")
sc.tl.umap(tam)

fig, ax = plt.subplots(figsize=(5, 5))

sc.pl.umap(
    tam,
    color="Macrophage_subtype",
    ax=ax,
    show=False,
    size=6,
    frameon=False,
    legend_loc="right margin"
)

add_dashed_ellipse_by_group(
    ax=ax,
    adata=tam,
    groupby="Macrophage_subtype",
    groups=["M1", "M2"],
    scale=1.4,
    min_cells=30
)

ax.set_title("")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.savefig(f"{OUTDIR}/Figure1G_TAM_M1_M2_UMAP.pdf")
plt.savefig(f"{OUTDIR}/Figure1G_TAM_M1_M2_UMAP.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Figure 1H. CNV score violin plot by cell type
# ============================================================

plot_df = adata_all.obs[["celltype", "CNV_score"]].copy()

order = [
    "B cell", "HPC-like", "Malignant cell",
    "T cell", "TAM", "unclassified"
]
order = [x for x in order if x in plot_df["celltype"].unique()]

fig, ax = plt.subplots(figsize=(5, 4))

sns.violinplot(
    data=plot_df,
    x="celltype",
    y="CNV_score",
    order=order,
    inner=None,
    cut=0,
    linewidth=0.6,
    ax=ax
)

sns.boxplot(
    data=plot_df,
    x="celltype",
    y="CNV_score",
    order=order,
    width=0.15,
    showcaps=False,
    boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 0.6},
    whiskerprops={"linewidth": 0.6},
    medianprops={"color": "black", "linewidth": 0.8},
    showfliers=False,
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("CNV score")
ax.set_title("Kruskal-Wallis, p < 2.2e-16")
ax.tick_params(axis="x", rotation=45)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTDIR}/Figure1H_CNV_score_violin.pdf")
plt.savefig(f"{OUTDIR}/Figure1H_CNV_score_violin.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Figure 1I. Marker gene dotplot
# ============================================================

marker_genes = [
    "MS4A1", "CD79A",
    "COL1A1", "DCN",
    "IL7R", "LTB",
    "NKG7", "CST7",
    "FOXP3", "IL2RA",
    "C1QC", "LST1",
    "APOE", "CD163",
    "PECAM1", "KDR",
    "MKI67", "TOP2A", "UBE2C",
    "VIM", "FN1",
    "HIF1A", "VEGFA", "CA9"
]

marker_genes = [g for g in marker_genes if g in adata_all.var_names]

dot_order = [
    "B cell",
    "CAF",
    "CD4 T",
    "CD8 T",
    "Cycling",
    "EMT",
    "HPC-like",
    "Hypoxia",
    "M1",
    "M2",
    "TEC",
    "Treg"
]

# 构建 Figure1I 用的精细注释
adata_all.obs["celltype_fine"] = adata_all.obs["celltype"].astype(str)

# 回填 T subtype
adata_all.obs.loc[t_cells.obs_names, "celltype_fine"] = t_cells.obs["T_subtype"]

# 回填 TAM subtype
adata_all.obs.loc[tam.obs_names, "celltype_fine"] = tam.obs["Macrophage_subtype"]

# malignant functional state 简单示例
mal = adata_all[adata_all.obs["celltype"] == "Malignant cell"].copy()

state_sets = {
    "Cycling_score": ["MKI67", "TOP2A", "UBE2C"],
    "EMT_score": ["VIM", "FN1", "COL1A1"],
    "Hypoxia_score": ["HIF1A", "VEGFA", "CA9"]
}

for score_name, genes in state_sets.items():
    genes = [g for g in genes if g in mal.var_names]
    if len(genes) > 0:
        sc.tl.score_genes(mal, genes, score_name=score_name)

state_cols = [c for c in state_sets.keys() if c in mal.obs.columns]
mal.obs["malignant_state"] = mal.obs[state_cols].idxmax(axis=1).replace({
    "Cycling_score": "Cycling",
    "EMT_score": "EMT",
    "Hypoxia_score": "Hypoxia"
})

adata_all.obs.loc[mal.obs_names, "celltype_fine"] = mal.obs["malignant_state"]

valid_groups = [
    g for g in dot_order
    if g in adata_all.obs["celltype_fine"].unique()
]

sc.pl.dotplot(
    adata_all,
    var_names=marker_genes,
    groupby="celltype_fine",
    categories_order=valid_groups,
    standard_scale="var",
    dot_max=0.75,
    figsize=(12, 4),
    show=False
)

plt.savefig(f"{OUTDIR}/Figure1I_marker_dotplot.pdf", bbox_inches="tight")
plt.savefig(f"{OUTDIR}/Figure1I_marker_dotplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Figure 1 | Step 9. 保存最终对象
# ============================================================

adata_all.write_h5ad(f"{OUTDIR}/adata_Figure1_integrated_annotated.h5ad")

adata_all.obs.to_csv(f"{OUTDIR}/Figure1_cell_metadata.csv")
print("Done.")